# Training a single backbone on the 120-class laboratory-field dataset

One notebook, run three times - once per backbone - by changing `BACKBONE` in the
configuration cell. This is how the reported models were produced: the same notebook was
re-used, with the backbone and its `preprocess_input` swapped each time.

Relative to the working copy, two things are different here:

* the backbone and its preprocessing are selected by one variable instead of by editing
  import lines, so the three runs are reproducible from the same file;
* the lines that write the class probabilities and the training history to CSV were
  commented out in the working copy and are enabled here. They are what produce
  `<tag>_pred.csv` and `<tag>_history.csv`.

Settings follow Section 3.2 of the paper: 224x224 input, the augmentation listed below,
a GAP -> 512 -> 512 -> 120 head with no dropout, all layers trainable, Adamax at 1e-4,
batch size 16, 30 epochs, seed 42, final-epoch weights.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os import listdir

from keras.preprocessing.image import ImageDataGenerator
from keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adamax
from sklearn import metrics
from sklearn.metrics import classification_report


## Configuration

`BACKBONE` is the only line to change between runs. Each backbone is paired with the
`preprocess_input` that its own ImageNet weights were trained with - ResNet50 expects
channel-mean subtraction, InceptionV3 expects [-1, 1], and the Keras EfficientNet models
expect raw [0, 255] because they rescale and normalise internally.


In [ ]:
BACKBONE = 'resnet50'          # 'resnet50' | 'inceptionv3' | 'efficientnetb0'

directory_root = 'AVI-Dataset_60_20_20'
BATCH_SIZE = 16
EPOCHS     = 30
INT        = 1e-4              # learning rate
SEED       = 42

if BACKBONE == 'resnet50':
    from tensorflow.keras.applications import ResNet50 as Backbone
    from keras.applications.resnet import preprocess_input
    TAG = 'resnet'
elif BACKBONE == 'inceptionv3':
    from tensorflow.keras.applications import InceptionV3 as Backbone
    from keras.applications.inception_v3 import preprocess_input
    TAG = 'inception'
elif BACKBONE == 'efficientnetb0':
    from tensorflow.keras.applications import EfficientNetB0 as Backbone
    from keras.applications.efficientnet import preprocess_input
    TAG = 'efficient'
else:
    raise ValueError(BACKBONE)

print('backbone:', BACKBONE, '| output prefix:', TAG)


## Data generators

Augmentation is applied to the training split only; validation and test images are just
resized and normalised. The test generator uses `shuffle=False`, so the row order of the
saved probabilities matches the sorted class/filename order used everywhere else.


In [ ]:
train_generator = ImageDataGenerator(rotation_range=90,
                                     brightness_range=[0.1, 0.7],
                                     width_shift_range=0.5,
                                     height_shift_range=0.5,
                                     horizontal_flip=True,
                                     vertical_flip=True,
                                     validation_split=0,
                                     preprocessing_function=preprocess_input)

val_generator  = ImageDataGenerator(preprocessing_function=preprocess_input)
test_generator = ImageDataGenerator(preprocessing_function=preprocess_input)


In [ ]:
train_data_dir      = directory_root + '/train'
validation_data_dir = directory_root + '/val'
test_data_dir       = directory_root + '/test'

class_subset = sorted(listdir(train_data_dir))

traingen = train_generator.flow_from_directory(train_data_dir,
                                               target_size=(224, 224),
                                               class_mode='categorical',
                                               classes=class_subset,
                                               batch_size=BATCH_SIZE,
                                               shuffle=True, seed=SEED)

validgen = val_generator.flow_from_directory(validation_data_dir,
                                             target_size=(224, 224),
                                             class_mode='categorical',
                                             classes=class_subset,
                                             batch_size=BATCH_SIZE,
                                             shuffle=True, seed=SEED)

testgen = test_generator.flow_from_directory(test_data_dir,
                                             target_size=(224, 224),
                                             class_mode='categorical',
                                             classes=class_subset,
                                             batch_size=BATCH_SIZE,
                                             shuffle=False, seed=SEED)

n_steps     = traingen.samples // BATCH_SIZE
n_val_steps = validgen.samples // BATCH_SIZE
n_classes   = len(traingen.class_indices)
print(f'{traingen.samples} train / {validgen.samples} val / {testgen.samples} test, {n_classes} classes')


## Model

ImageNet weights, the classification head of Section 3.2, and every layer trainable.


In [ ]:
base = Backbone(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

out = base.output
out = GlobalAveragePooling2D()(out)
out = Dense(512, activation='relu')(out)
out = Dense(512, activation='relu')(out)
predictions = Dense(n_classes, activation='softmax')(out)

model = Model(inputs=base.input, outputs=predictions)
model.compile(loss='categorical_crossentropy',
              optimizer=Adamax(learning_rate=INT),
              metrics=['accuracy'])
print(f'{model.count_params():,} parameters')


## Training


In [ ]:
history = model.fit(traingen,
                    batch_size=BATCH_SIZE,
                    validation_data=validgen,
                    steps_per_epoch=n_steps,
                    epochs=EPOCHS,
                    verbose=1)


### Save the training history

Per-epoch accuracy and loss for both splits. This is the source of the training curves in
Figure 6 - keeping it as a CSV means the figure can be redrawn without re-running the
notebook or reading numbers out of a saved image.


In [ ]:
hist = pd.DataFrame(history.history)
hist.insert(0, 'epoch', np.arange(1, len(hist) + 1))
hist.to_csv(f'{TAG}_history.csv', index=False)
print(f'wrote {TAG}_history.csv', hist.shape)
hist.tail()


## Test-set evaluation


In [ ]:
scores = model.evaluate(testgen)
print(f'Test accuracy: {100 * scores[1]:.2f}%')


### Save the class probabilities

One row per test image, one column per class, in the generator's order. These files are
the input to the weighted soft-voting notebook, which is why they hold probabilities
rather than predicted labels - the weighted average needs the full distribution.


In [ ]:
y_test = testgen.classes
y_prob = model.predict(testgen, verbose=1)

pd.DataFrame(y_prob).to_csv(f'{TAG}_pred.csv', index=False)
print(f'wrote {TAG}_pred.csv', y_prob.shape)

y_pred = np.argmax(y_prob, axis=1)
print(f'accuracy from the saved probabilities: {100 * (y_pred == y_test).mean():.2f}%')


### Per-class report and confusion matrix


In [ ]:
class_labels = list(traingen.class_indices.keys())

print(classification_report(y_test, y_pred, target_names=class_labels, zero_division=0))

report = classification_report(y_test, y_pred, target_names=class_labels,
                               output_dict=True, zero_division=0)
pd.DataFrame(report).transpose().to_csv(f'{TAG}_report.csv')

cm = metrics.confusion_matrix(y_test, y_pred)
pd.DataFrame(cm, index=class_labels, columns=class_labels).to_csv(f'{TAG}_confusion_matrix.csv')
print(f'wrote {TAG}_report.csv and {TAG}_confusion_matrix.csv')


### Training curves


In [ ]:
ep = hist['epoch']
plt.figure(figsize=(7, 4.5))
plt.plot(ep, hist['accuracy'],     label='Training accuracy')
plt.plot(ep, hist['val_accuracy'], label='Validation accuracy')
plt.plot(ep, hist['loss'],         label='Training loss')
plt.plot(ep, hist['val_loss'],     label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Accuracy / loss'); plt.ylim(0, 1.5)
plt.title(f'{BACKBONE}: training and validation accuracy and loss')
plt.legend(); plt.tight_layout()
plt.savefig(f'{TAG}_accuracy_loss.pdf')
plt.savefig(f'{TAG}_accuracy_loss.png', dpi=300)
plt.show()


## Files written

| File | Used for |
|---|---|
| `<tag>_pred.csv` | class probabilities on the test set - input to the weighted voting notebook |
| `<tag>_history.csv` | per-epoch accuracy and loss - the training curves of Figure 6 |
| `<tag>_report.csv` | per-class precision, recall and F1 |
| `<tag>_confusion_matrix.csv` | 120 x 120 confusion matrix |
| `<tag>_accuracy_loss.pdf` / `.png` | training-curve figure |

where `<tag>` is `resnet`, `inception` or `efficient`. Run the notebook once per backbone,
then open `weighted_voting.ipynb`, which reads the three `_pred.csv` files.


In [ ]:
for f in sorted(os.listdir('.')):
    if f.startswith(TAG):
        print(f'  {f:40s} {os.path.getsize(f) / 1e6:8.2f} MB')
